# 19 · An ensemble on the p scale, and whether its verdicts are stable (Phase 5)

**Goal.** Build the final system from the **original recipe** (the owner's decision after the 13-fold evidence): three seeds, combined on the val-human tail-probability ("p") scale of notebook 18, with the threshold rule that the declared Phase 4 choice rule picked for the original recipe (rule (d), notebook 18b). Then check (1) that the Phase 4 threshold still means what it should for an ensemble, and (2) that the ensemble's verdicts are more stable than a single model's. The notebook ends with the **exact table of the single final held-out check**, which is defined here and **not computed** (the project owner sees its definition first).

**Everything in sections 1-7 was written and committed to git before any Phase 5 analysis was run.** The GPU jobs that produce the data (two more seeds of the original recipe on the full training set; leave-one-prompt-out fold 11 with three seeds) are prerequisites, not analyses, and were started first. The held-out prompts and held-out generators are not scored anywhere in this notebook before section 7 is signed off.

## 1 · The three models and where their scores come from

| seed | model | val and test scores |
|---|---|---|
| 42 | `models/distilroberta_finetuned` (notebook 08, the existing model; first 512 tokens) | `results/15_scores.csv`, column `roberta`, **only the val and test rows are used** (that file also holds held-out rows from Phase 1; they are not read) |
| 43 | `models/distilroberta_orig_s43` (same recipe, trained for this phase on all 28,006 training essays) | `results/17_final/orig_s43_scores.csv` (val and test only) |
| 44 | `models/distilroberta_orig_s44` (same) | `results/17_final/orig_s44_scores.csv` (val and test only) |

The recipe (notebook 08): DistilRoBERTa, 2 epochs, lr 2e-5, 6% warm-up, 512 tokens, fp16, batch 16, best epoch by val AUC. No augmentation, no weights.

## 2 · The p scale per seed, the ensemble, and the combination rule (chosen on val by a rule declared here)

- **Per seed:** $p_s(x) = (1 + \#\{\text{val humans with score} \ge x\}) / (1 + n)$ with the 2,291 val humans (all 13 ordinary prompts), exactly as in notebook 18. Small = more AI-like.
- **Ensemble score:** $e(x) = \text{combine}(p_{42}(x), p_{43}(x), p_{44}(x))$ with *combine* either the **mean** or the **median** of the three p-values.
- **Ensemble p:** $e$ is not itself a p-value, so it is re-calibrated the same way: $p_{\text{ens}}(x) = (1 + \#\{\text{val humans } h : e(h) \le e(x)\}) / (1 + n)$. All thresholds $\tau$ below are on this scale, so the three seeds, the ensemble and any single model are on one scale.
- **Choice of the combination (val only, rule declared before computing):** the combination with the higher share of **val AI essays with $p_{\text{ens}} \le 0.01$** (val AUC and the share caught at the 1% level are the only val statistics used; the val AI essays are not used for calibration, so this is not circular). If the two differ by less than **0.2 percentage points** (about two of the 1,210 val AI essays), the **mean** is chosen (declared tie-break: the simpler one). `test` and the held-out sets play no part in this choice.

## 3 · Testing the Phase 4 assumption: does the threshold carry over from single models to the ensemble?

Phase 4's thresholds were derived from **single-seed** leave-one-prompt-out models. The assumption that carries them to the ensemble (notebook 18, section 7): on a new topic, human essays land in the low-$p$ tail zone **at the same rates for the ensemble as for a single model**. Testable on two folds with three seeds of the original recipe:

- **Fold 7 (Facial action coding system)**: seeds 42, 43, 44 from notebook 16b (`results/16b_lopo`).
- **Fold 11 (Seeking multiple opinions)**: seeds 42, 43, 44 from this phase (`results/19_phase5/lopo`), chosen *by a rule declared now*: the prompt with the highest false-flag rate for the original recipe (5.80% in 16b) that does not already have three seeds, i.e. where an inflation effect is largest and the test is most informative. (Fold 11 is also the direct check on regression to the mean: its original-recipe rate with other seeds.)

For each fold, with each seed's p of the left-out humans computed against *that seed's own* other-prompt val humans (the same essays for every seed), and the ensemble's $p_{\text{ens}}$ computed from the seeds' p-values with the combination chosen in section 2 and calibrated on the same other-prompt val humans:

- the **false-flag rate at $p \le \tau$** for $\tau = 0.01$ and $0.005$ (nominal rates), for each single seed and for the ensemble, with Wilson intervals, and the **inflation factor** = observed rate / $\tau$;
- the **paired difference** (ensemble minus the average of the three singles) in false-flag rate, with a cluster bootstrap interval (1,000 resamples, seed 42) that redraws the left-out humans and the val humans.

**Reading, declared:** the assumption **holds** for a fold if the paired difference's interval is below or contains zero (the ensemble's inflation is no larger than a single model's). It **fails** if the interval lies above zero (the ensemble flags more new-topic humans than a single model at the same $\tau$): then $\tau$ would be too lax for the ensemble and would have to be re-derived. Both folds are reported separately; two topics are a small basis, and I say so.

## 4 · Stability

All at the **final operating point** $\tau(1\%) = 0.00235$ (rule (d), notebook 18b); the band edge $\tau(5\%) = 0.01032$ is used for the typo test below.

**(i) Agreement of verdicts across seeds.** Among essays flagged by **at least one** of the three single seeds, the share flagged by **all three** (the statistic of 16b, where it was 28 of 72), on the val and test humans, the val and test AI essays, and the left-out humans of folds 7 and 11. For the ensemble the analogue uses the three **leave-one-seed-out sub-ensembles** (two seeds each). *Caveat declared in advance:* each pair of sub-ensembles shares a seed, which inflates their agreement; this measure therefore flatters the ensemble and is **an upper bound, not a fair comparison**. Pairwise Cohen's kappa is reported as well.

**(ii) Flip rate under 5% typos for essays near the threshold.** Each essay is scored again after `inject_typos(text, 0.05)` (two independent typo copies per essay). *Near the threshold* = the essay's own (unperturbed) $p$ lies within a **factor of 3 of $\tau(1\%)$** on either side ($\tau/3 \le p \le 3\tau$; for a single model $p$ is that model's, for the ensemble it is $p_{\text{ens}}$). A **flip** = the verdict (flagged / not flagged at $\tau(1\%)$) differs between the original essay and its typo copy. Reported: the flip rate among near-threshold essays and among all essays; the single models (average over the three seeds) against the ensemble; a paired bootstrap interval for their difference. Data: the val and test essays (the three saved final models, scored here) and the left-out essays of fold 11 (the three fold models, scored in `scripts/lopo_fold_typos.py`, because fold models are not kept). Fold 7 has no typo scores.

## 5 · The final system's operating points and zones

The final system is the ensemble of the three original-recipe seeds with the combination chosen in section 2, and the thresholds are the ones the declared choice rule picked for the original recipe (notebook 18b, rule (d), from all 13 topics): **$\tau(0.5\%) = 0.00093$, $\tau(1\%) = 0.00235$, $\tau(5\%) = 0.01032$**. Zones: **flagged** $p_{\text{ens}} \le \tau_{\text{flag}}$; **cannot tell** (to be worded as proposed in 18b, "unusual but common for student essays") $\tau_{\text{flag}} < p_{\text{ens}} \le \tau(5\%)$; **not flagged** above. Two operating points: **A**: $\tau_{\text{flag}} = \tau(1\%)$; **B**: $\tau_{\text{flag}} = \tau(0.5\%)$; the band's upper edge is $\tau(5\%)$ in both.

## 6 · The single final held-out check: the exact table (defined here, NOT computed)

**Rule of use.** Run **once**, at the end of this phase, for the **final ensemble + thresholds + band together**, after the project owner has seen this definition. No change to the ensemble, the combination, the thresholds or the band after it has been run. The held-out prompts ("Does the electoral college work?", "Summer projects") and the held-out generators (`darragh_claude_v6`, `darragh_claude_v7`, `falcon_180b_v1`) are scored by the three final models and combined exactly as in section 2 (the val-human calibration is the same; nothing is re-fitted).

**Table H1, human essays of each held-out prompt, separately** (rows: Electoral college; Summer projects; both pooled). For operating point A ($\tau_{\text{flag}} = \tau(1\%)$) and B ($\tau(0.5\%)$): *n*; **% flagged**, **% in the cannot-tell band** and **% not flagged**, each with a 95% Wilson interval (the pooled row: cluster bootstrap, 1,000 resamples, seed 42). Two blocks: all human essays, and on-topic human essays only.

**Table H2, AI essays of the held-out prompts** (same rows, same two operating points): **% flagged (AI caught)**, **% in the band**, **% missed**; all essays and on-topic only.

**Table H3, AI essays of the held-out generators** (rows: `darragh_claude_v6`, `darragh_claude_v7`, `falcon_180b_v1`, pooled): **% flagged**, **% in the band**, **% missed**, both operating points; all essays and on-topic only.

**Table H4, context (not part of the claims):** the same human false-flag and AI-caught quantities for the **single seed-42 model at the old logit thresholds of notebook 09** (4.67 for "1%", 7.80 for "0.5%"), for continuity with the numbers already in the README and model card.

**Table H5, the check against what Phase 4 predicted.** Registered now, from `results/18b_*` (original recipe, rule (d), leave-one-topic-out over 13 topics, single-seed models):

| quantity (operating point A, $\tau(1\%)$) | prediction |
|---|---|
| mean human false-flag rate over new topics | 0.58% (95% interval 0.44-0.88) |
| human false-flag rate of any one new topic | within the range of the 13 left-out topics: **0.00% to 1.74%** (worst topic); 23% of topics above 1% |
| AI essays caught, new topics | 98.78% (97.74-99.21) |
| share of human / AI essays in the cannot-tell band | 2.49% / 0.69% |

H5 reports, for each held-out prompt, the observed human false-flag rate against the 0.00-1.74% range and the AI caught against 97.7-99.2%, and says plainly where an observation falls outside what was predicted. Summer projects was the hardest unseen prompt for the single original model (7.7% at the old 1% threshold), so a value near or above the top of the range there would be a finding, not a surprise.

**Not part of the check and not computed:** anything that would let the held-out results feed back into a choice.

## 6b · The held-out check: additions approved by the project owner (definition FIXED here, before any held-out score is computed)

This cell completes the definition in section 6. It was written, and committed on its own, **before** any held-out essay was scored by the seed-43 and seed-44 models or by the ensemble. The run is `20_final_heldout_check.ipynb`, executed once.

### Preamble: what this check is and is not

- **The held-out prompts are not pristine.** The seed-42 model has been scored on them before (notebooks 08-14 and the README numbers), and an early choice, making the strict threshold the demo's default, was made after looking at them.
- **No choice since Phase 2 used them**: windowed scoring, the three variants, the 13 leave-one-prompt-out folds, the p scale, the threshold rules, the band edges, the ensemble combination and the thresholds were all chosen without them. The held-out prompts and generators were explicitly excluded from every selection.
- **Seeds 43 and 44 have never been scored on them.** This is the first time the final ensemble sees them.
- **Two prompts are two more topics.** The check tests Phase 4 (does the threshold hold on new topics?) on two more topics; it does **not replace** Phase 4's 13-topic evidence, and a result on two topics cannot overturn it.

### The final system and the operating points (as in section 5)

Ensemble = the **mean** of the three seeds' val-human tail probabilities, re-calibrated on the 2,291 val humans (`src/ensemble.py`); nothing is re-fitted. **Operating point A:** flag if p_ens <= tau(1%) = 0.00235, cannot tell up to tau(5%) = 0.01032. **Operating point B:** flag if p_ens <= tau(0.5%) = 0.00093, cannot tell up to tau(5%). Essays are scored on their first 512 tokens, as everywhere. Held-out essays: the 7,135 essays of `heldout_prompt` (4,464 human, 2,671 AI; two prompts) and the 2,721 AI essays of `heldout_generator` (`darragh_claude_v6`, `darragh_claude_v7`, `falcon_180b_v1`).

### Headline numbers (declared now; everything else is secondary)

1. **Pooled ON-TOPIC human essays of the two held-out prompts: % flagged at operating point A.**
2. **Pooled ON-TOPIC held-out-generator AI essays: % AI caught (flagged) at operating point A.**

### Failure criterion (declared now)

The 1% target is **missed** if the **lower end of the 95% interval of the pooled on-topic human % flagged at A is above 1%** (the interval of the pooled row: cluster bootstrap by `dup_cluster`, 1,000 resamples, seed 42; the Wilson interval is shown beside it for reference but the criterion uses the bootstrap one). Otherwise the target is **not missed** (which is not the same as "met": a point estimate above 1% with a lower end below 1% is reported as such). For each held-out prompt, its on-topic human % flagged at A is also placed **among the 13 left-out-topic rates** of rule (d) at tau(1%) (`results/18b_by_topic_1pct_original.csv`): its **rank** among the 14 values (1 = highest) and its **percentile** (the share of the 13 left-out rates below it).

### The tables

- **H1, human essays of each held-out prompt** (Electoral college; Summer projects; pooled). Operating points A and B: n, **% flagged, % cannot tell, % not flagged**, 95% Wilson intervals per prompt, cluster-bootstrap interval for the pooled row. Two blocks: all human essays and **on-topic** human essays (the headline is the on-topic pooled row at A).
- **H2, AI essays of the held-out prompts** (same rows, both operating points, both blocks): % flagged (AI caught), % cannot tell, % missed.
- **H3, AI essays of the held-out generators** (rows: `darragh_claude_v6`, `darragh_claude_v7`, `falcon_180b_v1`, pooled; both operating points; all and on-topic): % flagged, % cannot tell, % missed. The headline is the on-topic pooled row at A.
- **H4, a pipeline check, run FIRST.** The **seed-42 model** (`models/distilroberta_finetuned`), scored by the same code path as the other two seeds, at the old logit thresholds **4.666 ("1%") and 7.796 ("0.5%")** of notebook 09, on **on-topic** essays, must reproduce the numbers of `results/09_threshold_rates.csv`: **held-out-prompt human false flags 3.85% / 2.31%** and **held-out-generator AI caught 99.76% (99.8%) / 99.17% (99.2%)**. *Tolerance, declared now (my reading of "to rounding"):* each human rate within **2 essays** of the 09 count (fp16 batching can move an essay at the threshold) and each generator rate within **0.1 points**. **If H4 fails, the run stops and the project owner is told before anything is interpreted;** H1-H7 are not interpreted until H4 passes.
- **H5, against what Phase 4 predicted** (operating point A): each held-out prompt's on-topic human % flagged against the 13 left-out-topic rates (range 0.00-1.74%, rank, percentile as above) and against 23% of topics over target; AI caught against 97.7-99.2% (97.74-99.21 interval of 18b); the share of human and AI essays in the band against 2.49% and 0.69%.
- **H6, how much of each zone is AI.** P(AI | flagged) and P(AI | cannot tell) at AI shares of arriving essays of **35% (the dataset's 34.6%), 10% and 5%**, from the **held-out rates**: the human zone shares from the pooled on-topic held-out-prompt humans, the AI zone shares from the pooled on-topic AI essays of the held-out prompts and generators, Bayes' rule exactly as in 18b, operating point A, with the same cluster-bootstrap interval. **Proposed wording for the cannot-tell band** (not applied anywhere; from `18b_threshold_original_recipe.ipynb`, section 3 of its verdict): name the zone **"Unusual, but common for student essays"** and say *"Essays in this range are mostly written by students (in our tests about 7 in 8 when a third of essays are AI-written, and about 32 in 33 when 1 in 10 is). This is not evidence of AI writing."*; for the flagged zone: *"This reads like the AI essays we tested. If 1 in 10 essays in a class is AI-written, about 1 in 20 flagged essays is still a student's."* H6 reports whether the held-out numbers support those sentences or change them.
- **H7, agreement among flagged held-out humans.** Among the held-out human essays flagged by the **ensemble** at operating point A (per prompt and pooled, all and on-topic): how many are also flagged by **each single seed** (p_s <= tau(1%) with that seed's own val-human reference), and how many are flagged by **all three**.
- **H8, context only (not part of any claim): the single seed-42 model at the old thresholds** is H4 itself, extended to the all-essays block; it is shown for continuity with the README and model card.

### Outputs and the order of work

1. `results/20_heldout_scores.csv` (new): **per-essay held-out scores** (row id, split, label, source, prompt, on-topic flag, the three seeds' logits, the three seeds' p, p_ens, zones at A and at B) and `results/20_heldout_tables.csv`.
2. **The results are committed to git immediately after the run, before any interpretation is written.** The interpretation is a separate commit.
3. The notebook is run **once**. No change to the ensemble, the combination, the thresholds, the band, the tables or the criteria after it has been run.

## 7 · What is declared but not part of this notebook's claims

- The single-seed folds are the basis of the Phase 4 thresholds; the ensemble check in section 3 rests on **two topics** and three seeds each.
- The leave-one-seed-out sub-ensembles in 4(i) share seeds; the typo test in 4(ii) is the cleaner stability evidence.
- Seeds 43 and 44 of the full-training models are trained here; seed 42 is the existing model. There is no ensemble of more than three.

In [1]:
import os
os.environ['HF_HUB_OFFLINE'] = '1'
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
from src.data import load_splits
from src.evaluate import wilson_interval, ClusterSampler
from src.conformal import tail_p

SEED, N_BOOT = 42, 1000
TAU = {'0.5%': 0.00093, '1%': 0.00235, '5%': 0.01032}      # notebook 18b, rule (d), from all 13 topics (declared in section 5)
pd.set_option('display.width', 230); pd.set_option('display.max_colwidth', 100)
data = load_splits()

## 8 · The three models' scores and the combination (section 2 applied)

*(Everything from here on runs the analyses declared above; the declaration cells are unchanged.)*

In [2]:
from sklearn.metrics import roc_auc_score
from src.ensemble import PEnsemble, inflation_fold, agreement, flips, wilson_pct, combine
from src.conformal import tail_p

vt = data[data['split'].isin(['val', 'test'])]
s42 = pd.read_csv('../results/15_scores.csv', index_col='row_id')['roberta']                      # the existing model; ONLY val/test rows are used below
f43 = pd.read_csv('../results/17_final/orig_s43_scores.csv', index_col='row_id')
f44 = pd.read_csv('../results/17_final/orig_s44_scores.csv', index_col='row_id')
assert set(f43['split']) <= {'val', 'test'} and set(f44['split']) <= {'val', 'test'}
sc = pd.DataFrame({42: s42.loc[vt.index], 43: f43['logit'].loc[vt.index], 44: f44['logit'].loc[vt.index]})
assert not sc.isna().any().any()
rows = []
for s in sc.columns:
    v, t = vt['split'] == 'val', vt['split'] == 'test'
    rows.append({'seed': s, 'val AUC': roc_auc_score(vt.loc[v, 'label'], sc.loc[v, s]), 'test AUC': roc_auc_score(vt.loc[t, 'label'], sc.loc[t, s]),
                 'val 1% logit threshold': float(np.quantile(sc.loc[v & (vt['label'] == 0), s], 0.99))})
print('the three seeds of the original recipe (first 512 tokens); the logit thresholds differ, the p scale removes that:')
pd.DataFrame(rows).set_index('seed').round(5)

the three seeds of the original recipe (first 512 tokens); the logit thresholds differ, the p scale removes that:


,val AUC,test AUC,val 1% logit threshold
seed,,,
42,0.99972,0.99947,4.66602
43,0.99982,0.99981,5.27578
44,0.99977,0.99963,7.81523


In [3]:
# section 2: choose the combination on val only, by the rule declared in section 2
vh = vt[(vt['split'] == 'val') & (vt['label'] == 0)].index
va = vt[(vt['split'] == 'val') & (vt['label'] == 1)].index
val_ai = {}
for how in ('mean', 'median'):
    ens = PEnsemble(sc.loc[vh].to_numpy(), how)
    val_ai[how] = 100 * float((ens.p_ens(sc.loc[va].to_numpy()) <= 0.01).mean())
single = {s: 100 * float((tail_p(sc.loc[va, s].to_numpy(), sc.loc[vh, s].to_numpy()) <= 0.01).mean()) for s in sc.columns}
print('val AI essays with p <= 0.01 (of', len(va), '):', {k: round(v, 2) for k, v in val_ai.items()}, '| single seeds:', {k: round(v, 2) for k, v in single.items()})
HOW = 'mean' if abs(val_ai['mean'] - val_ai['median']) < 0.2 else max(val_ai, key=val_ai.get)
print(f"difference mean - median: {val_ai['mean'] - val_ai['median']:+.2f} points  ->  combination chosen by the declared rule: {HOW}")

val AI essays with p <= 0.01 (of 1210 ): {'mean': 99.59, 'median': 99.34} | single seeds: {42: 99.34, 43: 99.42, 44: 99.26}
difference mean - median: +0.25 points  ->  combination chosen by the declared rule: mean


## 9 · Does the Phase 4 threshold carry over to the ensemble? (section 3, folds 7 and 11)

*(The combination used is the one chosen above on val; nothing here was used to choose it.)*

In [4]:
FOLD_FILES = {7: ('Facial action coding system', '../results/16b_lopo/scores_fold07_seed{s}.csv'),
              11: ('Seeking multiple opinions', '../results/19_phase5/lopo/scores_fold11_seed{s}.csv')}
def fold_matrices(path_tpl, seeds=(42, 43, 44)):
    parts = {s: pd.read_csv(path_tpl.format(s=s), index_col='row_id').sort_index() for s in seeds}
    first = parts[seeds[0]]
    for s in seeds[1:]:
        assert first.index.equals(parts[s].index) and (first['role'].to_numpy() == parts[s]['role'].to_numpy()).all()
    vh_idx = first[(first['role'] == 'other_val') & (first['label'] == 0)].index
    lh_idx = first[(first['role'] == 'left_out') & (first['label'] == 0)].index
    la_idx = first[(first['role'] == 'left_out') & (first['label'] == 1)].index
    M = lambda idx: np.stack([parts[s].loc[idx, 'logit'].to_numpy() for s in seeds], axis=1)
    return dict(parts=parts, vh=vh_idx, lh=lh_idx, la=la_idx, val_h=M(vh_idx), left_h=M(lh_idx), left_a=M(la_idx))
inf_rows, FOLD = [], {}
for k, (name, tpl) in FOLD_FILES.items():
    FOLD[k] = fold_matrices(tpl)
    d = FOLD[k]
    r = inflation_fold(d['val_h'], d['left_h'], taus=(0.01, 0.005), how=HOW, val_clusters=data.loc[d['vh'], 'dup_cluster'].to_numpy(),
                       left_clusters=data.loc[d['lh'], 'dup_cluster'].to_numpy(), n_boot=N_BOOT, seed=SEED)
    inf_rows.append(r.assign(fold=k, topic=name))
inf = pd.concat(inf_rows, ignore_index=True)
inf.round(4).to_csv('../results/19_inflation_test.csv', index=False)
show = inf[['topic', 'tau', 'single seeds %', 'mean of singles %', 'ensemble %', 'inflation, singles (x nominal)', 'inflation, ensemble (x nominal)', 'ensemble minus singles (pts)', 'lo', 'hi', 'n']].copy()
show['assumption (section 3 reading)'] = np.where(show['hi'] < 0, 'holds (ensemble lower)', np.where(show['lo'] > 0, 'FAILS (ensemble higher)', 'holds (interval contains 0)'))
show.round(3)

,topic,tau,single seeds %,mean of singles %,ensemble %,"inflation, singles (x nominal)","inflation, ensemble (x nominal)",ensemble minus singles (pts),lo,hi,n,assumption (section 3 reading)
0,Facial action coding system,0.010,"[2.45, 1.85, 1.94]",2.077,2.446,2.077,2.446,0.369,-0.154,1.169,2167,holds (interval contains 0)
1,Facial action coding system,0.005,"[0.74, 0.55, 0.78]",0.692,0.877,1.384,1.754,0.185,-0.261,1.031,2167,holds (interval contains 0)
2,Seeking multiple opinions,0.010,"[5.99, 4.9, 5.28]",5.391,5.735,5.391,5.735,0.344,-0.688,1.804,1552,holds (interval contains 0)
3,Seeking multiple opinions,0.005,"[3.61, 2.13, 2.77]",2.835,3.222,5.670,6.443,0.387,-0.773,1.418,1552,holds (interval contains 0)


## 10 · Stability (section 4)

### (i) Agreement of verdicts across seeds, at tau(1%) = 0.00235

*Deviation from the declaration, stated plainly:* the declaration lists "val and test humans", but the val humans are the **calibration set of the p scale itself**, so their p-values are circular; only the **test humans** are used among humans (the AI essays of val and test are fine: the AI essays are not used for calibration).

In [5]:
TAU1 = TAU['1%']
def flags_for(X_single, ref_single, ref_how=HOW):
    """verdicts of each single seed (p_s <= tau) and of each leave-one-seed-out sub-ensemble (p_ens <= tau); X, ref: (n, 3) score matrices."""
    S = X_single.shape[1]
    ens_all = PEnsemble(ref_single, ref_how)
    single = ens_all.p_seeds(X_single) <= TAU1
    subs = []
    for drop in range(S):
        keep = [i for i in range(S) if i != drop]
        subs.append(PEnsemble(ref_single[:, keep], ref_how).p_ens(X_single[:, keep]) <= TAU1)
    return single, np.stack(subs, axis=1), ens_all.p_ens(X_single) <= TAU1
sets = {}
t_h = vt[(vt['split'] == 'test') & (vt['label'] == 0)].index; ai_idx = vt[(vt['label'] == 1)].index
ref_vt = sc.loc[vh].to_numpy()
sets['test humans'] = sc.loc[t_h].to_numpy(); sets['val + test AI essays'] = sc.loc[ai_idx].to_numpy()
refs = {'test humans': ref_vt, 'val + test AI essays': ref_vt}
for k, d in FOLD.items():
    sets[f'fold {k} left-out humans'] = d['left_h']; refs[f'fold {k} left-out humans'] = d['val_h']
rows = []
for name, X in sets.items():
    single, subs, full = flags_for(X, refs[name])
    a, b = agreement(single), agreement(subs)
    rows.append({'essays': name, 'n': len(X), 'single seeds: flagged by >=1': a['flagged by at least one'], 'by all 3': a['flagged by all'], 'share by all %': round(a['share flagged by all %'], 1), 'kappa': round(a['mean pairwise kappa'], 3),
                 'sub-ensembles: flagged by >=1': b['flagged by at least one'], 'sub-ensembles: by all 3': b['flagged by all'], 'share by all % (upper bound)': round(b['share flagged by all %'], 1), 'kappa (sub)': round(b['mean pairwise kappa'], 3)})
agree_tab = pd.DataFrame(rows).set_index('essays')
agree_tab.to_csv('../results/19_agreement.csv')
agree_tab

,n,single seeds: flagged by >=1,by all 3,share by all %,kappa,sub-ensembles: flagged by >=1,sub-ensembles: by all 3,share by all % (upper bound),kappa (sub)
essays,,,,,,,,,
test humans,2291,5,2,40.0,0.698,5,2,40.0,0.679
val + test AI essays,2420,2402,2392,99.6,0.855,2402,2396,99.8,0.903
fold 7 left-out humans,2167,7,1,14.3,0.349,6,1,16.7,0.444
fold 11 left-out humans,1552,44,7,15.9,0.487,28,10,35.7,0.650


### (ii) Flip rate under 5% typos for essays near the threshold

The test essays and the val AI essays are scored again by the three final models after `inject_typos(text, 0.05)` (two independent copies each); for fold 11 the left-out essays were scored after typos inside `scripts/lopo_fold_typos.py` (the fold models are not kept). Each perturbed essay's $p$ is computed against the same **unperturbed** reference humans.

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from src.paraphraser import inject_typos

TYPO_CACHE = '../results/19_phase5/typo_scores_test_valAI.csv'
eval_idx = vt[(vt['split'] == 'test') | ((vt['split'] == 'val') & (vt['label'] == 1))].index       # test essays + val AI essays (val humans are the reference)
if os.path.exists(TYPO_CACHE):
    typo = pd.read_csv(TYPO_CACHE, index_col='row_id')
else:
    typo = pd.DataFrame(index=eval_idx)
    texts = {c: [inject_typos(data.loc[i, 'text'], 0.05, seed=c * 1_000_003 + int(i)) for i in eval_idx] for c in (1, 2)}
    @torch.no_grad()
    def score(model_dir, text_list, batch_size=64):
        tok = AutoTokenizer.from_pretrained(model_dir); mdl = AutoModelForSequenceClassification.from_pretrained(model_dir).cuda().eval()
        ids = tok(text_list, truncation=True, max_length=512)['input_ids']
        out = np.empty(len(ids), dtype=np.float32); order = np.argsort([len(x) for x in ids], kind='stable')
        for s in range(0, len(order), batch_size):
            ch = order[s:s + batch_size]; w = max(len(ids[i]) for i in ch)
            x = torch.full((len(ch), w), tok.pad_token_id, dtype=torch.long)
            for r, i in enumerate(ch): x[r, :len(ids[i])] = torch.tensor(ids[i])
            with torch.autocast('cuda', dtype=torch.float16):
                out[ch] = mdl(input_ids=x.cuda(), attention_mask=(x != tok.pad_token_id).long().cuda()).logits.squeeze(1).float().cpu().numpy()
        del mdl; torch.cuda.empty_cache()
        return out
    for seed_, mdir in ((42, '../models/distilroberta_finetuned'), (43, '../models/distilroberta_orig_s43'), (44, '../models/distilroberta_orig_s44')):
        for c in (1, 2):
            typo[f's{seed_}_typo{c}'] = score(mdir, texts[c])
    typo.to_csv(TYPO_CACHE)
print(typo.shape)

(4711, 6)


In [7]:
def flip_table(name, ref, X0, Xt, how=HOW):
    """X0: (n, 3) clean logits; Xt: list of two (n, 3) typo-copy logits. Flip rates, singles (each seed's own near set) and the ensemble."""
    ens = PEnsemble(ref, how)
    p0s, p0e = ens.p_seeds(X0), ens.p_ens(X0)
    ptS = [ens.p_seeds(x) for x in Xt]; ptE = [ens.p_ens(x) for x in Xt]
    stats_ = {}
    for s in range(3):
        stats_[f'seed {s}'] = flips(p0s[:, s], [p[:, s] for p in ptS], TAU1)
    stats_['ensemble'] = flips(p0e, ptE, TAU1)
    return stats_
def summarise_flips(name, stats_, n_boot=N_BOOT):
    n = len(next(iter(stats_.values()))[0])
    rng = np.random.default_rng(SEED)
    def rate(rows, key):
        fl, near = stats_[key]
        nn = near[rows]
        return (fl[rows][nn].mean() if nn.any() else np.nan), (fl[rows].mean())
    out = {}
    for key in stats_:
        r_near, r_all = rate(np.arange(n), key)
        out[key] = {'near-threshold essays': int(stats_[key][1].sum()), 'flip rate near %': 100 * r_near, 'flip rate all essays %': 100 * r_all}
    d_near, d_all = [], []
    for _ in range(n_boot):
        rows = rng.integers(0, n, n)
        rs = [rate(rows, f'seed {s}') for s in range(3)]; re = rate(rows, 'ensemble')
        d_near.append(100 * (re[0] - np.nanmean([x[0] for x in rs]))); d_all.append(100 * (re[1] - np.mean([x[1] for x in rs])))
    tab = pd.DataFrame(out).T
    tab.loc['singles (mean of 3)'] = [tab.loc[[f'seed {s}' for s in range(3)], 'near-threshold essays'].mean(), tab.loc[[f'seed {s}' for s in range(3)], 'flip rate near %'].mean(), tab.loc[[f'seed {s}' for s in range(3)], 'flip rate all essays %'].mean()]
    lo_n, hi_n = np.nanpercentile(d_near, [2.5, 97.5]); lo_a, hi_a = np.percentile(d_all, [2.5, 97.5])
    print(f"{name}: ensemble minus singles, flip rate near the threshold {np.nanmean(d_near):+.2f} pts [{lo_n:+.2f}, {hi_n:+.2f}]; all essays {np.mean(d_all):+.3f} pts [{lo_a:+.3f}, {hi_a:+.3f}]")
    return tab.round(3)
# test essays + val AI essays (references: the val humans, unperturbed)
X0 = sc.loc[typo.index].to_numpy()
Xt = [typo[[f's{s}_typo{c}' for s in (42, 43, 44)]].to_numpy() for c in (1, 2)]
ft_vt = flip_table('test + val AI', ref_vt, X0, Xt)
flip_vt = summarise_flips('test essays + val AI essays', ft_vt)
flip_vt

test essays + val AI essays: ensemble minus singles, flip rate near the threshold +4.18 pts [-8.23, +17.21]; all essays -1.080 pts [-1.288, -0.860]


,near-threshold essays,flip rate near %,flip rate all essays %
seed 0,21.000,33.333,0.860
seed 1,26.000,26.923,0.488
seed 2,21.000,16.667,5.678
ensemble,32.000,29.688,1.263
singles (mean of 3),22.667,25.641,2.342


In [8]:
# fold 11: left-out humans and AI essays after typos (the three fold models of scripts/lopo_fold_typos.py)
d = FOLD[11]
lo_idx = d['parts'][42][d['parts'][42]['role'] == 'left_out'].index
X0 = np.stack([d['parts'][s].loc[lo_idx, 'logit'].to_numpy() for s in (42, 43, 44)], axis=1)
Xt = [np.stack([d['parts'][s].loc[lo_idx, f'logit_typo{c}'].to_numpy() for s in (42, 43, 44)], axis=1) for c in (1, 2)]
ft11 = flip_table('fold 11', d['val_h'], X0, Xt)
flip11 = summarise_flips('fold 11 left-out essays (humans and AI)', ft11)
pd.concat({'test + val AI': flip_vt, 'fold 11 left-out': flip11}).to_csv('../results/19_flip_rates.csv')
flip11

fold 11 left-out essays (humans and AI): ensemble minus singles, flip rate near the threshold -7.28 pts [-15.21, +0.45]; all essays -1.781 pts [-2.074, -1.506]


,near-threshold essays,flip rate near %,flip rate all essays %
seed 0,107.000,43.458,1.442
seed 1,62.000,40.323,1.331
seed 2,87.000,36.782,10.355
ensemble,94.000,32.979,2.601
singles (mean of 3),85.333,40.187,4.376


## 11 · Supplementary (not declared in sections 1-7; added after seeing the results above, and labelled as such)

Three descriptive checks that the declared tests raised: (a) why one single seed flips so often under typos, (b) the false-flag rates of the three-seed ensemble **at the final thresholds** (the declared inflation test used the nominal $\tau$ = 0.01 and 0.005; the final system uses $\tau(1\%) = 0.00235$ and $\tau(0.5\%) = 0.00093$), and (c) the original recipe's rate on fold 11 over four runs. No decision depends on them.

In [9]:
# (a) who flips: flip rate by label and the seed's cutoff in logits, test essays + val AI essays
typo_idx = typo.index
lab = vt.loc[typo_idx, 'label'].to_numpy()
ens_vt = PEnsemble(sc.loc[vh].to_numpy(), HOW)
p0s = ens_vt.p_seeds(sc.loc[typo_idx].to_numpy())
rows = []
for s, seed_ in enumerate((42, 43, 44)):
    pt = [ens_vt.p_seeds(typo[[f's{x}_typo{c}' for x in (42, 43, 44)]].to_numpy())[:, s] for c in (1, 2)]
    fl, near = flips(p0s[:, s], pt, TAU1)
    cut = np.sort(sc.loc[vh, seed_].to_numpy())[::-1][int(round(TAU1 * (len(vh) + 1))) - 1]          # the logit above which p <= tau(1%)
    rows.append({'seed': seed_, 'logit cutoff at tau(1%)': round(cut, 2), 'flip rate, human essays %': round(100 * fl[lab == 0].mean(), 2), 'flip rate, AI essays %': round(100 * fl[lab == 1].mean(), 2),
                 'AI logit median, clean': round(float(np.median(sc.loc[typo_idx, seed_][lab == 1])), 2), 'AI logit median, typo copy 1': round(float(np.median(typo[f's{seed_}_typo1'][lab == 1])), 2)})
pd.DataFrame(rows).set_index('seed')

,logit cutoff at tau(1%),"flip rate, human essays %","flip rate, AI essays %","AI logit median, clean","AI logit median, typo copy 1"
seed,,,,,
42,9.09,0.11,1.57,9.75,9.71
43,7.26,0.17,0.79,9.59,9.44
44,9.38,0.13,10.93,9.62,9.58


In [10]:
# (b) false flags on the two folds at the final thresholds, singles and the ensemble, against Phase 4's leave-one-topic-out prediction for that topic (rule (d), single-seed models)
PRED = {7: {'1%': 0.18, '0.5%': 0.14}, 11: {'1%': 1.74, '0.5%': 0.77}}        # from results/18b_by_topic_1pct_original.csv (1%) and the 0.5% by-topic values of rule (d)
rows = []
for k, (name, _) in FOLD_FILES.items():
    d = FOLD[k]; e = PEnsemble(d['val_h'], HOW)
    ps, ph, pa = e.p_seeds(d['left_h']), e.p_ens(d['left_h']), e.p_ens(d['left_a'])
    for tname, tau in (('1%', TAU['1%']), ('0.5%', TAU['0.5%'])):
        k_e = int((ph <= tau).sum())
        rows.append({'topic': name, 'threshold': f'tau({tname}) = {tau}', 'single seeds, humans flagged %': [round(100 * (ps[:, s] <= tau).mean(), 2) for s in range(3)],
                     'ensemble, humans flagged %': wilson_pct(k_e, len(ph)), 'ensemble, AI caught %': round(100 * float((pa <= tau).mean()), 1),
                     'Phase 4 prediction (single seed, this topic) %': PRED[k][tname]})
pd.DataFrame(rows).set_index(['topic', 'threshold'])

single seeds, humans flagged % ensemble, humans flagged %  ensemble, AI caught %  Phase 4 prediction (single seed, this topic) %
topic                       threshold                                                                                                                                           
Facial action coding system tau(1%) = 0.00235               [0.18, 0.05, 0.18]          0.09 [0.03, 0.34]                   99.6                                            0.18
                            tau(0.5%) = 0.00093              [0.14, 0.0, 0.05]          0.05 [0.01, 0.26]                   99.4                                            0.14
Seeking multiple opinions   tau(1%) = 0.00235               [2.51, 1.48, 0.84]          0.97 [0.59, 1.59]                   99.1                                            1.74
                            tau(0.5%) = 0.00093              [0.9, 0.52, 0.52]          0.39 [0.18, 0.84]                   98.9                                            0.77

In [11]:
# (c) fold 11, original recipe, four runs: the 16b run, the Phase 5 rerun of the same seed, and two more seeds
runs = pd.DataFrame({'run': ['16b, seed 42', 'Phase 5 rerun, seed 42 (identical recipe and seed)', 'Phase 5, seed 43', 'Phase 5, seed 44'],
                     'humans flagged': [90, 101, 76, 84]})
runs['of'] = 1552; runs['rate %'] = (100 * runs['humans flagged'] / runs['of']).round(2)
runs['Wilson'] = [wilson_pct(k, 1552).split(' ', 1)[1] for k in runs['humans flagged']]
print(f"mean {runs['rate %'].mean():.2f}%, SD {runs['rate %'].std(ddof=1):.2f} points, range {runs['rate %'].min()} to {runs['rate %'].max()}")
runs

mean 5.66%, SD 0.68 points, range 4.9 to 6.51


,run,humans flagged,of,rate %,Wilson
0,"16b, seed 42",90,1552,5.80,"[4.74, 7.07]"
1,"Phase 5 rerun, seed 42 (identical recipe and seed)",101,1552,6.51,"[5.38, 7.85]"
2,"Phase 5, seed 43",76,1552,4.90,"[3.93, 6.09]"
3,"Phase 5, seed 44",84,1552,5.41,"[4.39, 6.65]"


## Verdict

**1. The combination: mean.** On val, the share of AI essays with $p_{\text{ens}} \le 0.01$ is **99.59%** for the mean and **99.34%** for the median; the difference (+0.25 points) is above the declared 0.2-point margin, so the rule picks the **mean** (the same one the tie-break would have chosen). The three single seeds catch 99.34%, 99.42% and 99.26%.

**2. A threshold in logits again belongs to one model.** The three seeds of the original recipe have val AUC 0.99972, 0.99982, 0.99977 and val 1% thresholds of **+4.67, +5.28 and +7.82 logits**. On the p scale they agree by construction.

**3. The Phase 4 assumption is not contradicted, and the ensemble is no better than a single model at holding it.** (Declared test, two topics, nominal $\tau$.) On Facial action coding system at $\tau = 0.01$ the single seeds flag 2.45%, 1.85%, 1.94% of humans (mean 2.08%, 2.1 times the nominal 1%) and the ensemble 2.45%; ensemble minus singles **+0.37 points [-0.15, +1.17]**. On Seeking multiple opinions, singles 5.99%, 4.90%, 5.28% (mean 5.39%, 5.4 times nominal), ensemble 5.74%, **+0.34 [-0.69, +1.80]**. At $\tau = 0.005$ the differences are +0.19 [-0.26, +1.03] and +0.39 [-0.77, +1.42]. All four intervals contain zero, so by the declared reading the assumption **holds**; but all four point estimates are slightly *positive*, so there is **no evidence that averaging three seeds reduces the new-topic inflation**, which was the optimistic reading. Two topics are a small basis.
- **At the final thresholds** (supplementary, section 11): the ensemble flags 0.09% of Facial action coding humans at $\tau(1\%)$ (Phase 4 predicted 0.18% for that topic with single seeds) and **0.97% [0.59, 1.59]** of Seeking multiple opinions humans (predicted 1.74%; single seeds 2.51%, 1.48%, 0.84%), catching 99.6% and 99.1% of AI essays there. Both are consistent with the Phase 4 predictions.

**4. Individual verdicts are unstable across seeds, and a two-seed ensemble improves agreement only partly.** Among essays flagged by at least one of the three single seeds at $\tau(1\%)$, the share flagged by all three is **40% (2 of 5) on test humans**, **14% (1 of 7) on Facial action coding humans**, **16% (7 of 44) on Seeking multiple opinions humans**, and 99.6% on AI essays. The leave-one-seed-out two-seed ensembles agree more (40%, 17%, 36% on the same human sets), **an upper bound**, because every pair shares a seed. The same pattern as the 28 of 72 of 16b: the flagged tail of human essays is largely different from seed to seed.

**5. Flip rates under 5% typos.** Near the threshold ($\tau/3 \le p \le 3\tau$), **25.6% of the single models' near-threshold essays flip** on the test + val-AI set (ensemble 29.7%; difference +4.2 points [-8.2, +17.2]) and **40.2% on fold 11** (ensemble 33.0%; **-7.3 points [-15.2, +0.5]**): an essay near the line is close to a coin flip under a light perturbation, for single models and for the ensemble alike, and the ensemble's advantage near the line is not demonstrated (the intervals include zero; few essays are near the line). Over all essays the ensemble flips less than the average single model (**-1.1 points [-1.3, -0.9]** on test + val AI; **-1.8 points [-2.1, -1.5]** on fold 11), but this is driven by one seed: **seed 44 flips 5.7% of all essays (10.9% of AI essays)** because its cutoff at $\tau(1\%)$ is **9.38 logits** (seeds 42 and 43: 9.09 and 7.26), inside the saturated AI mass (AI median 9.6), so light typos nudge AI essays across it; for human essays all three seeds flip 0.1-0.2%. The ensemble is protected from that seed's behaviour by averaging, which is the real, if indirect, benefit.

**6. Regression to the mean is not supported on fold 11.** The original recipe's rate on Seeking multiple opinions (picked as an extreme at 5.80%) over four runs is **5.80%, 6.51%, 4.90%, 5.41%** (mean 5.66%, SD 0.68 points; the identical recipe and seed gave 5.80% and 6.51%). The fold's rate stayed high with other seeds, so for this fold the extreme was a topic effect, not noise. That weakens the earlier suggestion that regression to the mean explains the 7-fold result for variant A (`17b`, `17c`); the unexplained part is that variant A's gains on some prompts (Distance learning, Community service) were not reproducible on the six added prompts, one of which (Cowboy) it harmed badly. The run-to-run spread of one fold (0.7-1.6 points) is larger than the 0.34 points measured earlier on the median fold, so single-run differences below about 1.5 points on a high-rate prompt should not be over-read.

**Limits.** Two topics, three seeds each, for the ensemble test; the typo test has few near-threshold essays (21-107); the leave-one-seed-out agreement is an upper bound; the supplementary section was added after seeing results. The held-out prompts and generators were **not** scored.

## Key takeaways

- **Ensemble = the mean of the three seeds' val-human tail probabilities, re-calibrated on the val humans** (chosen on val: 99.59% against 99.34% of val AI at $p \le 0.01$). Thresholds: $\tau(0.5\%) = 0.00093$, $\tau(1\%) = 0.00235$, $\tau(5\%) = 0.01032$ (rule (d), notebook 18b).
- **The Phase 4 threshold carries over to the ensemble on the two topics tested** (no interval excludes zero; the ensemble does not reduce inflation). At the final thresholds the three-seed ensemble's false-flag rates on the two folds match what Phase 4 predicted.
- **Verdicts on flagged humans are unstable across seeds** (16-40% agreement), and **essays near the line flip under 5% typos about 30-40% of the time**, ensemble or not. A flag near the line is not a stable fact about an essay.
- **The ensemble's main benefit is robustness to a bad seed** (it flips 1.1-1.8 points less overall than the average single model, mostly by diluting seed 44), not a lower false-flag rate.
- **Regression to the mean is not supported on fold 11**: four runs of the original recipe give 4.9-6.5%.